<a href="https://colab.research.google.com/github/GOATIER69/DI_BOOCAMP/blob/main/DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def is_balanced(expression: str) -> bool:
    """
    Vérifie si une expression a des parenthèses équilibrées.

    Args:
        expression: Chaîne contenant l'expression à vérifier

    Returns:
        bool: True si les parenthèses sont équilibrées, False sinon
    """
    # Pile pour stocker les parenthèses ouvrantes
    stack = []

    # Dictionnaire de correspondance
    matching_pairs = {
        ')': '(',
        '}': '{',
        ']': '['
    }

    # Parcourir chaque caractère de l'expression
    for char in expression:
        # Si c'est une parenthèse ouvrante, on l'empile
        if char in '({[':
            stack.append(char)
        # Si c'est une parenthèse fermante
        elif char in ')}]':
            # Si la pile est vide, pas de parenthèse ouvrante correspondante
            if not stack:
                return False
            # Vérifier si la parenthèse fermante correspond à la dernière ouvrante
            if stack.pop() != matching_pairs[char]:
                return False

    # Si la pile est vide, toutes les parenthèses sont équilibrées
    return len(stack) == 0

# Tests
print("=== TESTS DE BASE ===")
print(f"'(1 + 2) * {[(3 / 4) - 5]}': {is_balanced('(1 + 2) * {[(3 / 4) - 5]}')}")  # True
print(f"'[1 + 2) * (3 - 4)]': {is_balanced('[1 + 2) * (3 - 4)]')}")                  # False
print(f"'((1 + 2)': {is_balanced('((1 + 2)')}")                                      # False
print(f"'()[]{{}}': {is_balanced('()[]{}')}")                                        # True
print(f"'([)]': {is_balanced('([)]')}")                                              # False

In [ ]:
def is_balanced_visual(expression: str) -> bool:
    """
    Version avec visualisation du processus de vérification.
    """
    stack = []
    matching_pairs = {')': '(', '}': '{', ']': '['}
    opening = '({['
    closing = ')}]'

    print(f"Analyse de l'expression: {expression}")
    print("=" * 60)

    for i, char in enumerate(expression):
        if char in opening:
            stack.append(char)
            print(f"{i:2d}: {char} → Empiler: {stack}")
        elif char in closing:
            if not stack:
                print(f"{i:2d}: {char} → ❌ Erreur: Parenthèse fermante sans ouvrante")
                return False
            last = stack.pop()
            if last != matching_pairs[char]:
                print(f"{i:2d}: {char} → ❌ Erreur: Mismatch '{last}' ≠ '{char}'")
                return False
            print(f"{i:2d}: {char} → Dépiler: {stack}")
        else:
            # Ignorer les autres caractères
            print(f"{i:2d}: {char} → Ignoré")

    print("-" * 60)
    if stack:
        print(f"❌ Parenthèses non fermées: {stack}")
        return False

    print("✅ Toutes les parenthèses sont équilibrées!")
    return True

# Test avec visualisation
print("\n=== VISUALISATION ===")
is_balanced_visual("(1 + 2) * {[(3 / 4) - 5]}")

In [ ]:
def is_balanced_extended(expression: str) -> bool:
    """
    Version étendue qui gère également les guillemets et apostrophes.

    Gère: (), {}, [], "", ''
    """
    stack = []

    # Dictionnaire de correspondance
    matching_pairs = {
        ')': '(',
        '}': '{',
        ']': '[',
        '"': '"',   # Les guillemets s'ouvrent et se ferment avec le même caractère
        "'": "'"    # Idem pour les apostrophes
    }

    # Caractères ouvrants
    opening = '({["\''
    # Caractères fermants
    closing = ')}]"\''

    # Pour gérer les guillemets qui s'ouvrent et se ferment avec le même caractère
    # On vérifie si le caractère est déjà dans la pile
    for char in expression:
        if char in opening:
            # Pour les guillemets et apostrophes, vérifier s'ils sont déjà ouverts
            if char in '"\'' and stack and stack[-1] == char:
                # C'est une fermeture
                stack.pop()
                continue
            # Sinon, c'est une ouverture
            stack.append(char)
        elif char in closing:
            if not stack:
                return False
            if stack.pop() != matching_pairs[char]:
                return False

    return len(stack) == 0

# Tests avec guillemets
print("\n=== TESTS AVEC GUILLEMETS ===")
test_cases = [
    ('(1 + 2) * "hello"', True),
    ('[1 + 2) * "hello"', False),
    ('"Hello (world)"', True),
    ("'Hello (world)'", True),
    ('"Hello (world', False),
    ("'Hello (world", False),
    ('([""])', True),
    ('(["]")', False),
]

for expr, expected in test_cases:
    result = is_balanced_extended(expr)
    status = "✅" if result == expected else "❌"
    print(f"{status} '{expr}' → {result} (attendu: {expected})")

In [ ]:
def check_balanced_advanced(expression: str) -> dict:
    """
    Version avancée qui donne des informations détaillées sur les erreurs.

    Returns:
        dict: {
            'balanced': bool,
            'errors': list,  # Liste des erreurs trouvées
            'unclosed': list,  # Parenthèses non fermées
            'unmatched': list  # Parenthèses fermantes sans ouvrante
        }
    """
    stack = []  # Chaque élément est un tuple (char, position)
    matching_pairs = {')': '(', '}': '{', ']': '[', '"': '"', "'": "'"}
    opening = '({["\''
    closing = ')}]"\''

    errors = []
    unmatched = []

    for i, char in enumerate(expression):
        if char in opening:
            # Pour les guillemets, vérifier s'ils sont déjà ouverts
            if char in '"\'' and stack and stack[-1][0] == char:
                # C'est une fermeture
                stack.pop()
                continue
            stack.append((char, i))
        elif char in closing:
            if not stack:
                errors.append(f"Position {i}: '{char}' fermant sans ouverture")
                unmatched.append((char, i))
                continue
            last, pos = stack.pop()
            if last != matching_pairs[char]:
                errors.append(f"Position {i}: '{char}' ne correspond pas à '{last}' à la position {pos}")
                # Remettre l'élément dans la pile pour continuer l'analyse
                stack.append((last, pos))

    unclosed = [(char, pos) for char, pos in stack]
    if unclosed:
        for char, pos in unclosed:
            errors.append(f"Position {pos}: '{char}' non fermé")

    return {
        'balanced': len(stack) == 0 and not errors,
        'errors': errors,
        'unclosed': unclosed,
        'unmatched': unmatched
    }

# Tests avec analyse détaillée
print("\n=== ANALYSE DÉTAILLÉE DES ERREURS ===")
expressions = [
    "(1 + 2) * {[(3 / 4) - 5]}",
    "[1 + 2) * (3 - 4)]",
    "((1 + 2)",
    "([)]",
    '"Hello (world"',
    "'Hello (world)"
]

for expr in expressions:
    print(f"\nExpression: {expr}")
    result = check_balanced_advanced(expr)
    print(f"Équilibrée: {result['balanced']}")
    if result['errors']:
        print("Erreurs:")
        for error in result['errors']:
            print(f"  ❌ {error}")
    else:
        print("  ✅ Aucune erreur")

In [ ]:
def is_balanced_with_validation(expression: str) -> bool:
    """
    Vérifie les parenthèses ET valide la syntaxe des opérateurs.
    """
    # D'abord vérifier les parenthèses
    if not is_balanced(expression):
        return False

    # Vérifier la syntaxe des opérateurs
    operators = '+-*/'
    for i, char in enumerate(expression):
        if char in operators:
            # Vérifier qu'il n'y a pas deux opérateurs consécutifs
            if i + 1 < len(expression) and expression[i + 1] in operators:
                return False
            # Vérifier que le premier caractère n'est pas un opérateur
            if i == 0:
                return False
            # Vérifier que le dernier caractère n'est pas un opérateur
            if i == len(expression) - 1:
                return False

    return True

print("\n=== VALIDATION DES OPÉRATEURS ===")
test_cases = [
    ("1 + 2", True),
    ("1 + + 2", False),
    ("+ 1 + 2", False),
    ("1 + 2 +", False),
    ("(1 + 2) * 3", True),
    ("(1 + + 2) * 3", False),
]

for expr, expected in test_cases:
    result = is_balanced_with_validation(expr)
    status = "✅" if result == expected else "❌"
    print(f"{status} '{expr}' → {result} (attendu: {expected})")

In [ ]:
class ParenthesisValidator:
    """
    Classe complète pour la validation des parenthèses avec options configurables.
    """

    def __init__(self, validate_quotes=False, validate_operators=False):
        """
        Initialise le validateur avec des options.

        Args:
            validate_quotes: Inclure les guillemets et apostrophes
            validate_operators: Valider la syntaxe des opérateurs
        """
        self.validate_quotes = validate_quotes
        self.validate_operators = validate_operators

        self.matching_pairs = {
            ')': '(', '}': '{', ']': '['
        }
        if validate_quotes:
            self.matching_pairs.update({
                '"': '"',
                "'": "'"
            })

        self.opening = ''.join(self.matching_pairs.values())
        self.closing = ''.join(self.matching_pairs.keys())

    def is_balanced(self, expression: str) -> bool:
        """
        Vérifie si l'expression a des parenthèses équilibrées.
        """
        stack = []

        for char in expression:
            if char in self.opening:
                # Gestion spéciale pour les guillemets
                if self.validate_quotes and char in '"\'' and stack and stack[-1] == char:
                    stack.pop()
                    continue
                stack.append(char)
            elif char in self.closing:
                if not stack:
                    return False
                if stack.pop() != self.matching_pairs[char]:
                    return False

        # Vérifier la syntaxe des opérateurs si demandé
        if self.validate_operators and len(stack) == 0:
            return self._validate_operators(expression)

        return len(stack) == 0

    def _validate_operators(self, expression: str) -> bool:
        """
        Valide la syntaxe des opérateurs.
        """
        operators = '+-*/'
        for i, char in enumerate(expression):
            if char in operators:
                if i == 0 or i == len(expression) - 1:
                    return False
                if expression[i + 1] in operators:
                    return False
        return True

    def get_error_report(self, expression: str) -> dict:
        """
        Retourne un rapport détaillé des erreurs.
        """
        return check_balanced_advanced(expression)

# Utilisation de la classe
print("\n=== VALIDATEUR AVEC OPTIONS ===")
validator = ParenthesisValidator(validate_quotes=True, validate_operators=True)

test_cases = [
    '(1 + 2) * "hello"',
    '(1 + + 2) * "hello"',
    '"Hello (world)"',
    '([""])',
]

for expr in test_cases:
    result = validator.is_balanced(expr)
    print(f"'{expr}' → {result}")

In [ ]:
def run_comprehensive_tests():
    """
    Exécute une batterie de tests complets.
    """
    test_suite = {
        "Tests de base": [
            ("()", True),
            ("[]", True),
            ("{}", True),
            ("(]", False),
            ("([)]", False),
            ("{[]}", True),
            ("", True),
        ],
        "Tests avec nombres et opérateurs": [
            ("(1 + 2)", True),
            ("(1 + 2] ", False),
            ("((1 + 2)", False),
            ("1 + (2 * 3)", True),
        ],
        "Tests complexes": [
            ("(1 + 2) * {[(3 / 4) - 5]}", True),
            ("[1 + 2) * (3 - 4)]", False),
            ("((1 + 2) * 3) + (4 - 5)", True),
            ("(1 + 2))", False),
            ("((1 + 2)", False),
        ],
        "Tests avec guillemets": [
            ('"hello"', True),
            ('(1 + "hello")', True),
            ('"hello', False),
            ('(1 + "hello)', False),
            ('("hello")', True),
        ],
        "Tests avec apostrophes": [
            ("'hello'", True),
            ("(1 + 'hello')", True),
            ("'hello", False),
        ],
        "Cas limites": [
            ("(", False),
            (")", False),
            ("()", True),
            ("(())", True),
            ("(()", False),
            ("())", False),
        ]
    }

    print("=" * 70)
    print("TESTS COMPLETS DU VALIDATEUR DE PARENTHÈSES")
    print("=" * 70)

    total_tests = 0
    passed_tests = 0

    for category, tests in test_suite.items():
        print(f"\n📋 {category}:")
        print("-" * 50)

        for expr, expected in tests:
            total_tests += 1
            result = is_balanced_extended(expr) if '"' in expr or "'" in expr else is_balanced(expr)
            status = "✅" if result == expected else "❌"

            if result == expected:
                passed_tests += 1

            print(f"{status} '{expr}' → {result} (attendu: {expected})")

    print("\n" + "=" * 70)
    print(f"RÉSULTATS: {passed_tests}/{total_tests} tests réussis")
    print(f"Taux de réussite: {passed_tests/total_tests*100:.1f}%")
    print("=" * 70)

run_comprehensive_tests()

In [ ]:
def interactive_validator():
    """
    Interface interactive pour tester le validateur.
    """
    print("\n" + "=" * 50)
    print("VALIDATEUR DE PARENTHÈSES - MODE INTERACTIF")
    print("=" * 50)
    print("Entrez 'quit' pour quitter")
    print("Entrez 'help' pour de l'aide")
    print()

    while True:
        expr = input("🔍 Expression à vérifier: ").strip()

        if expr.lower() == 'quit':
            print("👋 Au revoir!")
            break

        if expr.lower() == 'help':
            print("\n" + "-" * 40)
            print("Gère les parenthèses: () [] {}")
            print("Options avancées: guillemets \" et apostrophes '")
            print("Exemples valides:")
            print("  (1 + 2) * {[(3 / 4) - 5]}")
            print('  "Hello (world)"')
            print("  {[()]}")
            print("-" * 40)
            continue

        # Analyse
        result_basic = is_balanced(expr)
        result_extended = is_balanced_extended(expr)

        print("\n" + "-" * 40)
        print(f"📝 Expression: '{expr}'")
        print(f"🔷 Base (()[]{}): {'✅ Équilibrée' if result_basic else '❌ Non équilibrée'}")

        if '"' in expr or "'" in expr:
            print(f"🔷 Étendu (avec guillemets): {'✅ Équilibrée' if result_extended else '❌ Non équilibrée'}")

        # Analyse détaillée des erreurs
        details = check_balanced_advanced(expr)
        if details['errors']:
            print("\n⚠️ Erreurs détectées:")
            for error in details['errors']:
                print(f"  • {error}")
        else:
            print("✅ Aucune erreur détectée")

        print("-" * 40 + "\n")

# Lancer l'interface interactive
if __name__ == "__main__":
    interactive_validator()